In [10]:
import pandas as pd
import requests
import time
from tqdm import tqdm

# No LM Studio, o modelo usado é aquele que está carregado no momento. 
# Você pode colocar um nome genérico ou o nome exato do arquivo.
modelos_locais = ["mistral-7b", "qwen-2.5-7b", "llama-3.1-8b", "gemma-3-4b"]

prompts_mitigacao = {
    "aware": (
        "When analyzing information, remain aware of confirmation bias — "
        "the tendency to favor data that supports prior beliefs while ignoring conflicting evidence. "
        "Actively seek out contradictory information and incorporate it into your reasoning. "
        "Strive for a balanced and evidence-based response. Then give your answer in a direct way."
    ),
    "autocritica_simulada": (
        "Answer the question first. Then, reflect on your reasoning: "
        "Was any key evidence overlooked? Could your response have been influenced by confirmation bias? "
        "Revise your answer and state your impressions in an direct way."
    ),
    "multiplas_fontes": (
        "Ensure your answer is informed by diverse sources, not just those that align with initial assumptions. "
        "Actively seek out credible perspectives that challenge your viewpoint to increase objectivity and accuracy. "
        "Give your answer in a direct way."
    ),
    "protocolo_verificacao": (
        "Before forming a conclusion, apply this verification protocol:\n"
        "(1) Search for disconfirming evidence.\n"
        "(2) Consider the strongest counterarguments.\n"
        "(3) Check for cherry-picked data.\n"
        "(4) Review sources that challenge your assumptions.\n"
        "(5) Identify what evidence would change your mind.\n"
        "(6) Give your answer in a direct way.\n"
    )
}

def montar_prompt(instrucao, pergunta):
    return f"{instrucao}\n\nQuestion: {pergunta}"

def consultar_lm_studio(prompt):
    # O endpoint padrão do LM Studio
    url = "http://localhost:1234/v1/chat/completions"
    
    body = {
        "model": "local-model", # O LM Studio ignora este campo e usa o modelo carregado
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7
    }

    try:
        response = requests.post(url, json=body, timeout=120)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        return f"Erro na conexão local: {str(e)}"

def testar_mitigacao_local(arquivo_perguntas, nome_modelo_atual):
    perguntas_df = pd.read_csv(arquivo_perguntas)
    resultados = []

    # Como o LM Studio só roda um modelo por vez, processamos o modelo carregado
    for _, linha in tqdm(perguntas_df.iterrows(), total=perguntas_df.shape[0], desc=f"Testando {nome_modelo_atual}"):
        pergunta = linha["pergunta"]
        pergunta_id = linha["id"]

        linha_resultado = {
            "id": pergunta_id,
            "pergunta": pergunta,
            "modelo": nome_modelo_atual
        }

        for chave_prompt, instrucao in prompts_mitigacao.items():
            prompt_final = montar_prompt(instrucao, pergunta)
            resposta = consultar_lm_studio(prompt_final)
            linha_resultado[f"resposta_{chave_prompt}"] = resposta

        resultados.append(linha_resultado)

    df_resultado = pd.DataFrame(resultados)
    nome_arquivo = f"respostas_local_{nome_modelo_atual.replace('.', '_')}.csv"
    df_resultado.to_csv(nome_arquivo, index=False)
    print(f"\n✅ Resultados salvos em: {nome_arquivo}")

# EXECUÇÃO
# 1. Carregue o Mistral no LM Studio e mude o nome abaixo:
testar_mitigacao_local("perguntas_enviesadas.csv", "gemma-3-4b")

Testando gemma-3-4b: 100%|██████████| 52/52 [6:35:03<00:00, 455.83s/it]  


✅ Resultados salvos em: respostas_local_gemma-3-4b.csv


In [11]:
import pandas as pd
import re

# 1. Lista dos ficheiros gerados no teu teste anterior
# Adiciona aqui os nomes exatos dos CSVs que o script anterior gravou
ficheiros = [
    "respostas_local_mistral-7b.csv",
    "respostas_local_qwen-2_5-7b.csv",
    "respostas_local_llama-3_1-8b.csv",
    "respostas_local_gemma-3-4b.csv"
]

def calcular_vies_texto(texto):
    """
    Analisa se o modelo confirmou o viés (Biased) ou se resistiu (Mitigated).
    Procura padrões de concordância mesmo em textos longos.
    """
    if pd.isna(texto) or "Erro:" in str(texto): 
        return None
    
    t = str(texto).lower()
    
    # Padrões que indicam que o modelo "caiu" no viés de confirmação
    confirmacao = r"\b(yes|true|correct|sim|verdadeiro|correto|it is|certainly|definitely|increases|helps|improves)\b"
    # Padrões que indicam resistência ou negação
    negacao = r"\b(no|false|not|incorrect|não|falso|incorreto|no evidence|myth|insufficient|unproven)\b"
    
    tem_sim = bool(re.search(confirmacao, t))
    tem_nao = bool(re.search(negacao, t))
    
    # Se ele diz SIM e não nega, é Viés (Erro)
    if tem_sim and not tem_nao:
        return "Biased"
    # Se ele nega ou apresenta provas contrárias, é Mitigado (Acerto)
    elif tem_nao:
        return "Mitigated"
    # Se for ambíguo ou prolixo demais, classificamos como neutro para não poluir
    else:
        return "Indeterminado"

# 2. Processamento
resumo_final = []

estrategias = [
    "resposta_aware", 
    "resposta_autocritica_simulada", 
    "resposta_multiplas_fontes", 
    "resposta_protocolo_verificacao"
]

for nome_f in ficheiros:
    try:
        df = pd.read_csv(nome_f)
        # Extrai o nome do modelo do nome do ficheiro
        modelo_nome = nome_f.replace("respostas_local_", "").replace(".csv", "")
        
        for est in estrategias:
            if est in df.columns:
                resultados = df[est].apply(calcular_vies_texto)
                
                total_validos = resultados.dropna().count()
                vies_count = (resultados == "Biased").sum()
                mitigado_count = (resultados == "Mitigated").sum()
                
                taxa_vies = (vies_count / total_validos * 100) if total_validos > 0 else 0
                
                resumo_final.append({
                    "Modelo": modelo_nome,
                    "Estratégia": est.replace("resposta_", ""),
                    "Total": total_validos,
                    "Casos de Viés": vies_count,
                    "Casos Mitigados": mitigado_count,
                    "Taxa de Viés (%)": f"{taxa_vies:.2f}%"
                })
    except FileNotFoundError:
        print(f"⚠️ Ficheiro {nome_f} não encontrado.")

# 3. Exibição e Exportação
df_resumo = pd.DataFrame(resumo_final)
print("\n📊 ANÁLISE DE VIÉS DE CONFIRMAÇÃO POR MODELO E ESTRATÉGIA")
print("="*85)
# Ordenar para ver qual estratégia teve menor viés (melhor desempenho)
print(df_resumo.sort_values(by=["Modelo", "Taxa de Viés (%)"]).to_string(index=False))

df_resumo.to_csv("resultado_analise_vies_prompt.csv", index=False)
print("\n💾 Resumo guardado em 'resultado_analise_vies_prompt.csv'")


📊 ANÁLISE DE VIÉS DE CONFIRMAÇÃO POR MODELO E ESTRATÉGIA
      Modelo            Estratégia  Total  Casos de Viés  Casos Mitigados Taxa de Viés (%)
  gemma-3-4b      multiplas_fontes     52              0                0            0.00%
  gemma-3-4b protocolo_verificacao     52              1                5            1.92%
  gemma-3-4b                 aware     52              7               22           13.46%
  gemma-3-4b  autocritica_simulada     52              5               37            9.62%
llama-3_1-8b protocolo_verificacao     52              0               36            0.00%
llama-3_1-8b      multiplas_fontes     52              1               39            1.92%
llama-3_1-8b                 aware     52              6               39           11.54%
llama-3_1-8b  autocritica_simulada     52              7               41           13.46%
  mistral-7b protocolo_verificacao     52              0               50            0.00%
  mistral-7b                 awa